In [1]:
!pip install openai-agents

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 18.2 MB/s eta 0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 17.2 MB/s eta 0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.5 MB 11.4 MB/s eta 0:00:01
   ------------- -------------------------- 3.1/9.5 MB 7.7 MB/s eta 0:00:01
   ----------------- ---------------------- 4.2/9.5 MB 6.4 MB/s eta 0:00:01
   --

In [2]:
import os
import json
import asyncio
import requests

from typing import List
from agents import Agent, Runner, function_tool, trace

from google.colab import userdata

SECTORS_API_KEY = userdata.get('SECTORS_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

headers = {"Authorization": SECTORS_API_KEY}

ModuleNotFoundError: No module named 'google'

In [ ]:
def retrieve_from_endpoint(url: str) -> dict:

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as err:
        raise SystemExit(err)
    return json.dumps(data)

In [ ]:
@function_tool
def get_company_overview(ticker: str, country: str) -> str | None:
    """
    Get company overview from Singapore Exchange (SGX) or Indonesia Exchange (IDX)
    """
    assert country.lower() in ["indonesia", "singapore", "malaysia"], "Country must be either Indonesia, Singapore, or Malaysia"

    if(country.lower() == "indonesia"):
        url = f"https://api.sectors.app/v1/company/report/{ticker}/?sections=overview"
    if(country.lower() == "singapore"):
        url = f"https://api.sectors.app/v1/sgx/company/report/{ticker}/"
    if(country.lower() == "malaysia"):
        url = f"https://api.sectors.app/v1/klse/company/report/{ticker}/"

    try:
        return retrieve_from_endpoint(url)
    except Exception as e:
        print(f"Error occurred: {e}")
        return None

@function_tool
def get_top_companies_ranked(dimension: str) -> List[str]:
    """
    Return a list of top companies (symbol) based on certain dimension (dividend yield, total dividend, revenue, earnings, market_cap, PB ratio, PE ratio, or PS ratio)
    """

    url = f"https://api.sectors.app/v1/companies/top/?classifications={dimension}&n_stock=3"

    return retrieve_from_endpoint(url)

@function_tool
def find_companies_screener(query: str) -> List[str]:
    """
    Docstring for find_companies_screener

    :param query: natural language query to search for companies on Indonesia Stock Exchange (IDX)
    :type query: str
    :return: List of companies that match the stock screener query (IDX Stocks only)
    :rtype: List[str]
    """

    url = f"https://api.sectors.app/v2/companies/?q={query}"
    return retrieve_from_endpoint(url)

In [ ]:
natural_language_screener = Agent(
    name="Natural Language Screener",
    tools=[find_companies_screener],
    # complete the Agent creation process here
)

# create more than 1 Agent
# research_writer_agent = Agent(...)

# EXAMPLE PROMPTS. Create your own query in query3.
# query0 = "Find me a list of IDX banks with the largest dividend payout in q3 2025"
# query1 = "Tell me about the company listed on Singapore Exchange with ticker 'D05'."
query2 = "Screen for IDX companies where Prajogo Pangestu is a major shareholder."
query3 = ""

async def main():
  """
  CHALLENGE:
  Modify the code here to use a Multi-Agent workflow to have another
  Agent generate a list of companies that you want to research, based on some
  screening condition, then delegate to the second Agent to do the work
  """
  screener_assistant = await Runner.run(
    natural_language_screener, query2
  )
  print(screener_assistant.final_output)

await main()

Here are the IDX-listed companies where Prajogo Pangestu is a major shareholder:

1. PT Barito Pacific Tbk (BRPT.JK)
   - Prajogo Pangestu owns 71.37% of the shares.

2. PT Petrindo Jaya Kreasi Tbk (CUAN.JK)
   - Prajogo Pangestu owns 84.09% of the shares.

3. PT Chandra Asri Pacific Tbk (TPIA.JK)
   - Prajogo Pangestu owns 5.03% of the shares.

4. PT Barito Renewables Energy Tbk (BREN.JK)
   - Prajogo Pangestu owns 0.11% of the shares (directly, but the company is majority-owned by entities also linked to him).

Let me know if you want more detail on any of these companies or their shareholding structures!
